# RadCluster_1_0 — Cluster Dynamics for Irradiated Materials

A full per-size **cluster-dynamics** code for self-interstitial atom (SIA) loops, vacancy / He-vacancy cavities, and free interstitial helium in bcc Fe and EUROFER97. The code implements:

- **Master equations** (Eqs. 152, 155, 157) for SIA clusters $c_n$, He-vacancy bubbles $c_{m,\ell}$, and free helium $c_h$.
- **He-reduction** schemes: Case 2 — fission (Eq. 175, decoupled scalar $\ell_{\rm tot}$) and Case 1 — fusion (Eq. 174, mean-field $\bar\ell(m)$ per cavity class).
- **Size-bin moments** (Chapter 9, Eqs. 188–211) for large-$N$ efficiency, with three intra-bin shape closures.
- **Solute trapping** in EUROFER97 (Cr, W, Mn, C, N) via effective diffusivities (Eqs. 42, 48, 52).
- **1D / 3D mixed transport** for glissile SIA clusters (Eq. 141).

> **Primary reference** — Ghoniem, N.M. (2026), *"A Cluster Dynamics Model for Radiation Damage Evolution in Ferritic-Martensitic Steels"*, [`docs/Formulation/Defect_Cluster_Dynamics.pdf`](../../../docs/Formulation/Defect_Cluster_Dynamics.pdf). All equation numbers in this notebook refer to that document.

---

## A. Solver and physics options

### Solver modes

| `solver_mode` | Description |
|---|---|
| `full_system` | C++ CVODE BDF on the full state vector. Linear solver: dense / band / GMRES. |
| `active_window` | Two independent sliding windows (one over SIA size, one over VAC size) + OpenMP-parallel RHS. Thread count is auto-picked from $N_{\rm eq}$ (override via `OMP_NUM_THREADS`); when OpenMP is unavailable or only one thread is selected, the same code path runs serial. |

(Legacy aliases `cpp_full` and `sliding_OpenMP` are still accepted silently for back-compatibility.)

### Physics axes

The physics is selected by two **orthogonal** switches:

| Axis | Values | Meaning |
|---|---|---|
| `EQUATIONS` | `discrete` \| `bin_moment` | Per-size ODEs (Eqs. 152, 155, 157) vs. Chapter 9 bin-moment grouping (Eqs. 188–211). |
| `CASCADE`   | `fission`  \| `fusion`     | `fission` → He Case 2, Eq. 175 (decoupled, Table 2 fission spectrum). `fusion` → He Case 1, Eq. 174 (mean-field, Table 2 fusion spectrum). |

The four combinations map to the canonical `physics_option` strings used internally and on disk (the historical `full_CD` token is preserved in the combined string):

| EQUATIONS | CASCADE | `physics_option` |
|---|---|---|
| `discrete`   | `fission` | `full_CD_fission` |
| `discrete`   | `fusion`  | `full_CD_fusion` |
| `bin_moment` | `fission` | `bin_moment_CD_fission` |
| `bin_moment` | `fusion`  | `bin_moment_CD_fusion` |

`RadClusterSimulation` accepts either the new `(equations, cascade)` pair or the legacy `physics_option=` kwarg. Legacy `equations='full_CD'` is silently aliased to `'discrete'`.

---

## B. Main features (v1.0)

| Feature | Reference | Description |
|---|---|---|
| Two equation formulations | Eqs. 152, 155, 157 / Chapter 9 | *discrete* per-size ODEs or *bin-moment* logarithmic grouping |
| Two cascade source models | Eqs. 174, 175 / Table 2 | *fission* (Case 2 — decoupled scalar He) or *fusion* (Case 1 — mean-field He per class) |
| Hybrid discrete + binned grids | Eq. 188 (bin boundaries) | small clusters tracked individually, large clusters in geometric bins |
| Three intra-bin shape closures | Eqs. 195, 198, 204 | piecewise-constant ($P{=}1$), hat-function ($P{=}2$), log-normal ($P{=}3$) |
| EUROFER solute trapping | Eqs. 42, 48, 52 / Table 16 | Cr, W, Mn, C, N effective diffusivities |
| 1D / 3D mixed transport | Eqs. 140, 141 | glissile SIA clusters with mean-free-path correction |
| Cascade survival + clustering | Eqs. G_$\eta$, $\epsilon_m$ / Table 2 | fission and fusion parameter sets |
| C++ SUNDIALS / CVODE BDF solver | — | dense, banded, or GMRES linear solver |
| Two solver modes | — | `full_system` or `active_window` (sliding windows + OpenMP) |
| Two preconditioners | — | `Jacobi` (diagonal) or **`Woodbury`** (bordered-banded SMW, default for GMRES) |
| Adaptive domain doubling | — | $I$ and $V$ grow on the fly when the tail reaches the boundary |
| Excel-driven inputs | — | three-sheet workbook in `input/`; runtime overrides in this notebook |
| Conservation diagnostics | Eqs. 164, 165 | $\delta_{\rm FP}$ (Frenkel-pair) and $\delta_{\rm He}$ (helium) |
| Timestamped outputs | — | `output/YYYYMMDD_HHMMSS_<git-hash>/` with provenance, `results.pkl`, summary CSV, plots |

---

## C. Control variables

The variables below appear in the main script cell in the same order. Sizes are atom counts; concentrations are atomic fractions; energies in eV; temperature in K.

### C.1 Run-mode switches

| Variable | Type / values | Meaning |
|---|---|---|
| `DEBUG` | `True` / `False` | If `True`, all solver messages, override applications, and per-step diagnostics are printed. If `False`, solver chatter is captured silently. |
| `SOLVER_MODE` | `'full_system'` \| `'active_window'` | Time-stepping strategy (see §A). |
| `EQUATIONS` | `'discrete'` \| `'bin_moment'` | Per-size ODEs (Eqs. 152, 155, 157) vs. Chapter 9 bin-moment grouping. |
| `CASCADE` | `'fission'` \| `'fusion'` | Cascade spectrum + He-coupling case. `fission` → Eq. 175 (Case 2). `fusion` → Eq. 174 (Case 1). |
| `HE_KINETICS` | `'quasi_steady_state'` \| `'dynamic'` | QSS eliminates the free-He variable from the state vector (recommended). `dynamic` keeps $c_h$ as an explicit ODE (Eq. 157). |

### C.2 Cluster domain and mobility

| Variable | Typical | Meaning |
|---|---|---|
| `I` | 1e5 | Maximum SIA cluster size (largest $n$ tracked). Grows adaptively if `max_doublings > 0`. |
| `V` | 1e5 | Maximum vacancy / cavity cluster size. |
| `i_mobile` | 11–15 | Largest SIA cluster that can diffuse (1D glide). Larger clusters are sessile loops. |
| `v_mobile` | 2–5 | Largest mobile vacancy cluster (3D random walk). Larger cavities are sessile voids. |

### C.3 Discrete / bin-moment split  (`EQUATIONS = 'bin_moment'`)

| Variable | Default | Meaning |
|---|---|---|
| `i_discrete` | `2 * i_mobile` | Sizes $1 \ldots i_{\rm discrete}$ tracked as individual ODEs. |
| `v_discrete` | `2 * v_mobile` | Sizes $1 \ldots v_{\rm discrete}$ tracked as individual ODEs. |
| `I_bin` | 50 | Number of logarithmic bins covering $i_{\rm discrete}{+}1 \ldots I$. `None` for auto-pick from $r \approx 2$ (Eq. 189). |
| `V_bin` | 50 | Number of logarithmic bins covering $v_{\rm discrete}{+}1 \ldots V$. `None` for auto-pick. |
| `shape_function` | `'linear'` | Intra-bin shape closure: `'constant'` ($P{=}1$, Eq. 195), `'linear'` ($P{=}2$, hat, Eq. 198), `'lognormal'` ($P{=}3$, Eq. 204). |

The total number of ODEs for `bin_moment` + `fission` + QSS is

$$N_{\rm eq} = i_{\rm discrete} + P\,I_{\rm bin} + v_{\rm discrete} + P\,V_{\rm bin} + 1$$

with $P = 1, 2, 3$ for constant / linear / log-normal. Comparison at $I = V = 10^5$, $i_{\rm discrete} = v_{\rm discrete} = 30$, $P = 2$, $I_{\rm bin} = V_{\rm bin} = 50$:

| Mode | $N_{\rm eq}$ |
|---|---|
| Hybrid bin-moment | $\approx 261$ |
| `discrete` (full per-size) | $I + V + 2 = 200{,}002$ |

To **disable bin-moments and recover full per-size ODEs**, set `EQUATIONS = 'discrete'`, `I_bin = 0`, `V_bin = 0`, `i_discrete = I`, `v_discrete = V`.

### C.4 Numerical tolerances

| Variable | Default | Meaning |
|---|---|---|
| `C_FLOOR` | 1e-25 | Concentration floor (any $c_n$ below this is treated as zero). |
| `SOLVER_CONFIG['t_span']` | (1e-6, 1e2) s | Initial and final integration time. |
| `SOLVER_CONFIG['n_points']` | 200 | Number of output time points. |
| `SOLVER_CONFIG['log_time']` | `True` | Logarithmic spacing of output times. |
| `SOLVER_CONFIG['rtol']` | 1e-6 | CVODE relative tolerance. |
| `SOLVER_CONFIG['atol']` | 1e-20 | CVODE absolute tolerance. |

### C.5 Linear solver and preconditioner

| Key | Values | Meaning |
|---|---|---|
| `solver_method.linsol` | `'dense'` \| `'band'` \| `'gmres'` | Inner linear solver inside CVODE BDF. Use `band` for moderate $N_{\rm eq}$, `gmres` once $N_{\rm eq} \gtrsim 500$. |
| `solver_method.preconditioner` | `'Jacobi'` \| `'Woodbury'` | GMRES preconditioner. **Woodbury** exploits the bordered-banded Jacobian $J = T + UV^T$ with banded $T$ (half-bandwidth $\max(2 i_{\rm mobile}, 2 v_{\rm mobile})+1$) and rank-$r$ correction ($r = i_{\rm mobile} + v_{\rm mobile}$). Used only in `full_system`; sliding windows fall back to Jacobi automatically. |

### C.6 Sliding-window parameters  (`SOLVER_MODE = 'active_window'`)

| Key | Default | Meaning |
|---|---|---|
| `window_width` | 50 | Initial window width (shared by SIA and VAC) in size space. |
| `concentration_threshold` | 1e-18 | Concentration threshold that triggers a window-edge expansion. |
| `window_pad` | 100 | Sizes added when the window expands. |

### C.7 Physics overrides (`PARAM_OVERRIDES`)

Anything set here replaces the value read from `input/input_parameters.xlsx`. The override key is the *Symbol* column from the relevant Excel sheet. Set `PARAM_OVERRIDES = {}` to use Excel defaults verbatim. Common keys:

| Sheet | Key | Default | Meaning (eq. ref.) |
|---|---|---|---|
| Production | `eta` | 0.30 | Cascade survival efficiency $\eta$ (Eq. G_$\eta$, Table 2). |
| Production | `f_cl_i` | 0.58 | Fraction of SIAs born inside cascade clusters (Table 2). |
| Production | `f_cl_v` | 0.15 | Fraction of vacancies born inside cascade clusters (Table 2). |
| Diffusion | `E_m_1D` | 0.34 eV | 1D SIA cluster migration energy (Eq. 141). |
| Diffusion | `i_mobile` | 15 | SIA mobility cutoff (must match top-level value). |
| Diffusion | `L_hat` | 50 | Dimensionless mean free path $\hat L = L/a$ for 1D glide (Eq. P6). |
| Diffusion | `c_C` | 5e-4 | Dissolved C concentration (atom fraction, Table 16). |
| Diffusion | `E_b_C_SIA` | 0.45 eV | Carbon–SIA binding energy (Table 16). |
| Reactions | `rho_d` | 1e13 m⁻² | Dislocation density. |
| Reactions | `Z_i` | 1.1 | SIA dislocation bias factor (Table 26). |
| Reactions | `Z_ii` | 1.0 | SIA–SIA coalescence bias factor. |
| Reactions | `shape_function` | `'linear'` | Intra-bin shape closure (see §C.3). |

The script searches every input sheet for the key, so any other Excel parameter (e.g. temperature `T`, dose rate `G`, grain size `d_g`) can be overridden the same way.

---

## D. Typical example setups

Each example shows only the variables that change from the defaults in the main script cell.

### Example A — Quick smoke test (small system, fission, fast)

Useful to confirm the build, the Excel input, and the output pipeline are all wired up.

```python
SOLVER_MODE = 'full_system'
EQUATIONS   = 'discrete'        # full per-size (Eqs. 152, 155, 157)
CASCADE     = 'fission'         # Case 2, Eq. 175

I, V                     = int(1e3), int(1e3)
i_mobile, v_mobile       = 11, 5
i_discrete, v_discrete   = I, V
I_bin, V_bin             = 0, 0   # → recovers full_CD limit

SOLVER_CONFIG['t_span']                       = (1e-6, 1e0)
SOLVER_CONFIG['solver_method']['linsol']      = 'band'
PARAM_OVERRIDES = {}
```

### Example B — Production fission run (large domain, hybrid grid)

Best balance of accuracy and speed for long-time, high-dose runs.

```python
SOLVER_MODE = 'active_window'
EQUATIONS   = 'bin_moment'      # Chapter 9, Eqs. 188–211
CASCADE     = 'fission'         # Case 2, Eq. 175

I, V                     = int(1e6), int(1e6)
i_mobile, v_mobile       = 15, 5
i_discrete, v_discrete   = 2*i_mobile, 2*v_mobile
I_bin, V_bin             = 50, 50

SOLVER_CONFIG['t_span']                          = (1e-6, 1e8)   # ~100 dpa
SOLVER_CONFIG['solver_method']['linsol']         = 'gmres'
SOLVER_CONFIG['solver_method']['preconditioner'] = 'Woodbury'

PARAM_OVERRIDES = {
    'rho_d':           1e14,         # well-annealed steel
    'shape_function':  'lognormal',  # P=3 closure (Eq. 204)
}
```

### Example C — Fusion reactor cascade spectrum

Switches the cascade spectrum and He coupling to fusion-relevant values (Table 2).

```python
SOLVER_MODE = 'full_system'
EQUATIONS   = 'bin_moment'
CASCADE     = 'fusion'          # Case 1, Eq. 174

I, V                     = int(5e5), int(5e5)
i_mobile, v_mobile       = 15, 5
i_discrete, v_discrete   = 30, 10
I_bin, V_bin             = 60, 60

SOLVER_CONFIG['t_span'] = (1e-6, 3e7)

PARAM_OVERRIDES = {
    'eta':            0.28,          # fusion survival (Table 2)
    'f_cl_i':         0.65,
    'f_cl_v':         0.20,
    'shape_function': 'lognormal',
}
```

### Example D — Sensitivity sweep on dislocation density

Re-run the cell three times changing only `rho_d`. Each run produces its own timestamped output directory, so results are not overwritten.

```python
EQUATIONS    = 'bin_moment'
CASCADE      = 'fission'
I = V        = int(1e5)
I_bin = V_bin = 40

PARAM_OVERRIDES = {'rho_d': 1e13}   # then 1e14, then 1e15
```

### Example E — Solute-trapping study (high-C EUROFER variant)

Probes the impact of dissolved-C trapping (Eqs. 42, 48, 52) on SIA mobility and loop nucleation.

```python
EQUATIONS = 'bin_moment'
CASCADE   = 'fission'

PARAM_OVERRIDES = {
    'c_C':        2e-3,    # 4× nominal C (Table 16)
    'E_b_C_SIA':  0.65,    # stronger SIA–C binding
}
```

### Example F — Debug a stalled run

```python
DEBUG = True                                            # show every solver message
SOLVER_CONFIG['rtol'] = 1e-4                            # loosen tolerance
SOLVER_CONFIG['atol'] = 1e-18
SOLVER_CONFIG['solver_method']['linsol'] = 'dense'      # rule out GMRES issues
```

---

## E. What gets saved

After each run, results land in:

```
output/YYYYMMDD_HHMMSS_<git-hash>/
├── provenance.md       # input parameters, git SHA, solver settings
├── results.pkl         # full ODE solution (binary, NumPy-loadable)
├── summary.csv         # tabulated macroscopic quantities vs. time
└── plots/              # PNG figures (size distributions, swelling, He, …)
```

The end-of-cell summary prints the final dose, swelling, total He, mean SIA and vacancy cluster sizes, loop and void number densities, and the two conservation diagnostics:

- $\delta_{\rm FP}$ — Frenkel-pair conservation (Eq. 164)
- $\delta_{\rm He}$ — helium conservation (Eq. 165)

Both should remain $\lesssim 10^{-6}$ for a clean run; values above $10^{-3}$ indicate a coding or input error.

---

## F. Where to look next

- Permanent material / environment changes → edit `input/input_parameters.xlsx` (3 sheets: `Material_Environment`, `Physical_Properties`, `Model_Parameters`).
- Full equation reference → [`docs/Formulation/Defect_Cluster_Dynamics.pdf`](../../../docs/Formulation/Defect_Cluster_Dynamics.pdf).
- Implementation notes and equation index → [`RadCluster_1_0/CLAUDE.md`](../../CLAUDE.md).
- Diagnostic / development notebooks → `code/development/`.
- After editing C++ sources → set `FORCE_REBUILD = True` in the build cell below and re-run it.

## Build C++ solver (run once after code changes)

In [ ]:
import sys, os, subprocess
from pathlib import Path

# Add RadCluster_1_0 root to path
MODULE_ROOT = Path('../..').resolve()
REPO_ROOT   = MODULE_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
if str(MODULE_ROOT) not in sys.path:
    sys.path.insert(0, str(MODULE_ROOT))

print(f'Module root: {MODULE_ROOT}')
print(f'Repo root:   {REPO_ROOT}')

# ---------------------------------------------------------------------------
# Build the C++ SUNDIALS solver only if the platform-correct binary is
# missing. This makes the notebook portable across macOS / Linux / Windows:
# moving between OSes leaves the wrong-platform binary in build/ and the
# Python bridge cannot launch it, so we rebuild on first run for the new OS.
# Set FORCE_REBUILD = True to force a rebuild after C++ source changes.
# ---------------------------------------------------------------------------
FORCE_REBUILD = False

build_dir = MODULE_ROOT / 'build'
exe_name  = 'solver.exe' if sys.platform == 'win32' else 'solver'
exe_candidates = [
    build_dir / 'Release' / exe_name,
    build_dir / 'Debug'   / exe_name,
    build_dir              / exe_name,
]
exe_found = next((p for p in exe_candidates if p.exists()), None)

if exe_found is not None and not FORCE_REBUILD:
    print(f'C++ solver already built: {exe_found}')
    print('Set FORCE_REBUILD = True to rebuild after C++ source changes.')
else:
    if exe_found is None:
        print(f'No {exe_name} found under {build_dir} -- building now ...')
    else:
        print(f'FORCE_REBUILD set -- rebuilding {exe_found} ...')

    build_dir.mkdir(exist_ok=True)
    cmake_src  = MODULE_ROOT / 'cpp_utils'
    cmake_cmd  = ['cmake', '-S', str(cmake_src), '-B', str(build_dir),
                   '-DCMAKE_BUILD_TYPE=Release']
    build_cmd  = ['cmake', '--build', str(build_dir), '--config', 'Release']

    for cmd in [cmake_cmd, build_cmd]:
        res = subprocess.run(cmd, capture_output=True, text=True)
        print(' '.join(cmd))
        if res.returncode != 0:
            print('STDERR:', res.stderr[-2000:])
            raise RuntimeError(f'Build step failed: {cmd[0]} {cmd[1]}')
        else:
            print('OK')
            print(res.stdout[-500:])

    exe_found = next((p for p in exe_candidates if p.exists()), None)
    if exe_found is None:
        raise RuntimeError(f'Build completed but no {exe_name} found under {build_dir}')
    print(f'Built: {exe_found}')


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ══════════════════════════════════════════════════════════════════════════════
import sys, os, io
# Drop machine-scope OMP_NUM_THREADS so the C++ solver auto-picks threads from N_eq
os.environ.pop('OMP_NUM_THREADS', None)
from pathlib import Path

MODULE_ROOT = Path('../..').resolve()
REPO_ROOT   = MODULE_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
if str(MODULE_ROOT) not in sys.path:
    sys.path.insert(0, str(MODULE_ROOT))

import numpy as np
import importlib

# Reload all py_utils modules to pick up code changes
import RadCluster_1_0.py_utils.defect_production as _dp_mod
import RadCluster_1_0.py_utils.binding_energies as _be_mod
import RadCluster_1_0.py_utils.bin_moment_rates as _bmr
import RadCluster_1_0.py_utils.input_data as _inp_mod
import RadCluster_1_0.py_utils.reaction_rates as _rr_mod
import RadCluster_1_0.py_utils.rate_equations as _re_mod
import RadCluster_1_0.py_utils.cpp_bridge as _cb_mod
import RadCluster_1_0.py_utils.post_process as _pp_mod
import RadCluster_1_0.py_utils.simulation as _sim_mod
import RadCluster_1_0.py_utils.visualization as viz
for _m in [_dp_mod, _be_mod, _bmr, _inp_mod, _rr_mod, _re_mod,
           _cb_mod, _pp_mod, _sim_mod, viz]:
    importlib.reload(_m)
from RadCluster_1_0.py_utils.simulation import RadClusterSimulation

# ══════════════════════════════════════════════════════════════════════════════
# 1. Plot axis controls (override viz defaults; None = auto)
# ══════════════════════════════════════════════════════════════════════════════
# Three groups cover the whole plot suite:
#   concentration — concentration vs dose (point defects, totals, He, number
#                    densities, SIA/cavity cluster concentrations).
#   scalar        — scalar metrics vs dose (swelling, mean sizes, fraction
#                    breakdowns, conservation diagnostics).
#   size_dist     — concentration vs cluster size n/m or diameter (nm).
# Set xscale/yscale to 'log' or 'linear' (None = keep each plot's default),
# and xlim/ylim tuples to clip axes (use None on either side for auto).
# legend_fontsize controls the figure-level legends underneath cluster plots.
# ══════════════════════════════════════════════════════════════════════════════
PLOT_CONFIG = {
    'concentration': {
        'xlim':   (None, None),   # (xmin, xmax) in dpa; None = auto
        'ylim':   (1e10, None),   # (ymin, ymax) in m^-3; None = auto
        'xscale': None,           # 'log' | 'linear' | None=keep default
        'yscale': None,
    },
    'scalar': {
        'xlim':   (None, None),
        'ylim':   (None, None),
        'xscale': None,
        'yscale': None,
    },
    'size_dist': {
        'xlim':   (None, None),   # cluster size n/m or diameter (nm)
        'ylim':   (1e10, None),   # concentration in m^-3
        'xscale': None,
        'yscale': None,
    },
    'legend_fontsize': 5,         # font size of legends underneath plots
}
viz.set_plot_config(PLOT_CONFIG)


# ══════════════════════════════════════════════════════════════════════════════
# 2. User selections — cluster domain and mobility
# ══════════════════════════════════════════════════════════════════════════════
DEBUG = False
SOLVER_MODE = 'full_system' # "full_system" or "active_window" (auto-serial when threads=1)

# Two-axis physics selection (combined → physics_option string internally)
EQUATIONS = 'discrete'    # 'discrete' or 'bin_moment'
CASCADE   = 'fission'       # 'fission' or 'fusion'
SHAPE_FUNCTION = 'linear'   # 'constant', 'linear', or 'lognormal' (bin-moment closure)

# ── Domain sizes ─────────────────────────────────────────────────────────────
I          = int(1e3)    # max SIA cluster size  (grows adaptively if doublings > 0)
V          = int(1e3)    # max vacancy cluster size

# ── Mobility cutoffs ────────────────────────────────────────────────────────
i_mobile   = 1          # max mobile SIA cluster size
v_mobile   = 1          # max mobile vacancy cluster size

# ── Discrete / binned split ──────────────────────────────────────────────────
# Sizes 1..i_discrete tracked as individual ODEs; beyond that, bin moments.
# Defaults: i_discrete = i_mobile, v_discrete = v_mobile.
# Set I_bin = 0 and i_discrete = I to recover full_CD limit.

i_discrete = I    # max discrete SIA size
v_discrete = V    # max discrete vacancy size
I_bin      = 0         # SIA bin-moment equations (None = auto from r≈2)
V_bin      = 0         # VAC bin-moment equations (None = auto from r≈2)

# ── Other settings ───────────────────────────────────────────────────────────
C_FLOOR    = 1e-25       # concentration floor [atom fraction]
HE_KINETICS = 'quasi_steady_state' # "quasi_steady_state", "dynamic"


# ══════════════════════════════════════════════════════════════════════════════
# 3. Solver configuration
# OpenMP threads: auto-picked by the C++ solver from N_eq when
# OMP_NUM_THREADS is unset (popped above).  Set OMP_NUM_THREADS to override.
# Falls back to a single thread when OpenMP is not available.

# ══════════════════════════════════════════════════════════════════════════════
SOLVER_CONFIG = {
    't_span':   (1e-6, 1e3),
    'n_points': 200,
    'log_time': True,
    'rtol':     1e-6,
    'atol':     1e-20,
    'solver_method': {
        'linsol':                 'gmres',  # 'dense', 'band', or 'gmres'
        'preconditioner':        'Jacobi', # 'Jacobi' or 'Woodbury'
        'window_width':          10,
        'concentration_threshold': 1e-22,
        'window_pad':            20,
    }
}


# ══════════════════════════════════════════════════════════════════════════════
# 4. Initialize simulation (reads Excel file)
# ══════════════════════════════════════════════════════════════════════════════

_saved_out, _saved_err = sys.stdout, sys.stderr
if not DEBUG:
    sys.stdout = sys.stderr = io.StringIO()
try:
    sim = RadClusterSimulation(
        I=I, V=V,
        solver_mode=SOLVER_MODE,
        equations=EQUATIONS,
        cascade=CASCADE,
        C_floor=C_FLOOR,
        he_kinetics=HE_KINETICS,
        i_mobile=i_mobile,
        v_mobile=v_mobile,
    )
finally:
    sys.stdout, sys.stderr = _saved_out, _saved_err


# ══════════════════════════════════════════════════════════════════════════════
# 4b. Parameter overrides — applied AFTER reading Excel, BEFORE solver run.
#     Keys are the Symbol names from each Excel sheet.
#     Set PARAM_OVERRIDES = {} to use the Excel defaults.
# ══════════════════════════════════════════════════════════════════════════════

PARAM_OVERRIDES = {
    # ── Reactions sheet — irradiation conditions ─────────────────────────
    'T':         673,        # irradiation temperature [K] (default 573)

    # ── Production sheet (fission) ────────────────────────────────────────
    'eta':       0.3,        # cascade survival efficiency (default 0.30)
    'f_cl_i':    0.5,       # SIA clustering fraction    (default 0.58)
    'f_cl_v':    0.15,       # vacancy clustering fraction (default 0.15)

    # ── Diffusion sheet ──────────────────────────────────────────────────
    'E_m_1D':    0.4,       # 1D SIA cluster migration energy [eV] (default 0.34)
    'L_hat':     5,       # mean free path L/a (default 50)
    'c_C':       1e-3,    # dissolved C concentration [at/at] (default 5e-4)
    'E_b_C_SIA': 0.65,       # C-SIA binding energy [eV] (default 0.45)

    # ── Reactions sheet ──────────────────────────────────────────────────
    'rho_d':     1e14,       # dislocation density [m^-2] (default 1e13)
    'Z_i':       1.08,        # SIA dislocation bias factor (default 1.1)
    'Z_ii':      1.01,        # SIA-SIA coalescence bias factor (default 1.0)
}

# Inject discrete/bin controls and shape_function into overrides
PARAM_OVERRIDES['i_discrete']     = i_discrete
PARAM_OVERRIDES['v_discrete']     = v_discrete
PARAM_OVERRIDES['shape_function'] = SHAPE_FUNCTION
if I_bin is not None:
    PARAM_OVERRIDES['I_bin'] = I_bin
if V_bin is not None:
    PARAM_OVERRIDES['V_bin'] = V_bin

if PARAM_OVERRIDES:
    _saved_out2, _saved_err2 = sys.stdout, sys.stderr
    if not DEBUG:
        sys.stdout = sys.stderr = io.StringIO()
    try:
        inp = sim.input_data
        for key, val in PARAM_OVERRIDES.items():
            placed = False
            for d in [inp.production_fission, inp.production_fusion,
                      inp.diffusion, inp.reactions,
                      inp.energetics, inp.dissociation]:
                if key in d:
                    d[key] = val
                    placed = True
            if not placed:
                inp.reactions[key] = val
        # Mobility cutoffs need to land in `diffusion` (where _calculate_derived
        # reads them) AND in `reactions` (provenance), with int cast.
        for mob_key in ('i_mobile', 'v_mobile'):
            if mob_key in PARAM_OVERRIDES:
                v = int(PARAM_OVERRIDES[mob_key])
                inp.diffusion[mob_key] = v
                inp.reactions[mob_key] = v
        inp._calculate_derived()
        sim.rebuild_rates()
    finally:
        sys.stdout, sys.stderr = _saved_out2, _saved_err2

    print(f'Applied {len(PARAM_OVERRIDES)} parameter overrides:')
    for k, v in PARAM_OVERRIDES.items():
        print(f'  {k:>12} = {v}')
else:
    print('Using Excel defaults (no overrides)')

# ── Report system size ────────────────────────────────────────────────────
re = sim.rate_equations
if hasattr(re, 'I_bin'):
    P = re.n_mom
    print(f'\nHybrid discrete + bin-moment system (shape_function={re.shape_function!r}):')
    print(f'  Discrete SIA:  i = 1..{re.i_discrete}  ({re.i_discrete} ODEs)')
    print(f'  Binned SIA:    I_bin = {re.I_bin}  ({P} moments each = {P*re.I_bin} ODEs)')
    print(f'  Discrete VAC:  v = 1..{re.v_discrete}  ({re.v_discrete} ODEs)')
    print(f'  Binned VAC:    V_bin = {re.V_bin}  ({P} moments each = {P*re.V_bin} ODEs)')
    he_odes = re.N_eq - (re.i_discrete + P*re.I_bin + re.v_discrete + P*re.V_bin)
    print(f'  He state:      {he_odes} ODE(s)')
    print(f'  ──────────────────────')
    print(f'  Total N_eq = {re.N_eq}  (full_CD would be {sim.input_data.I + sim.input_data.V + 2})')
else:
    print(f'\nFull per-size system: N_eq = {re.N_eq}')


# ══════════════════════════════════════════════════════════════════════════════
# 5. Build live progress callback
# ══════════════════════════════════════════════════════════════════════════════
sim._progress_rows = []

rr      = sim.reaction_rates
rate_eq = sim.rate_equations
G_dpa   = sim.input_data.derived['G']
Omega   = sim.input_data.derived['Omega']
s2m     = 1.0 / Omega

_row_idx = [0]

def _progress_callback(row):
    sim._progress_rows.append(dict(row))
    j   = _row_idx[0];  _row_idx[0] += 1
    t_  = row.get('t', 0.0)
    dos = t_ * G_dpa
    if DEBUG:
        ci1 = row.get('c_i1', 0.0)
        cv1 = row.get('c_v1', 0.0)
        sys.stderr.write(
            f'  [{j:>4d}] t={t_:.4e}  dose={dos:.3e}'
            f'  Ci1={ci1*s2m:.3e}  Cv1={cv1*s2m:.3e} m^-3\n')
        sys.stderr.flush()


# ══════════════════════════════════════════════════════════════════════════════
# 6. Run simulation — TRULY ADAPTIVE CONTINUATION
#    Integration runs in short segments (points_per_segment output points
#    each).  After every segment the tail fraction is checked.  When the
#    threshold is first exceeded, I and/or V are doubled and
#    integration CONTINUES from that time point — no restart from t=0.
#    I and V have independent doubling budgets.
#
#    Graceful interrupt: press Ctrl+C (or Jupyter "Interrupt Kernel") at
#    any time.  The C++ subprocess is terminated, all completed segments
#    are kept, and results / plots are saved for the last integration point.
# ══════════════════════════════════════════════════════════════════════════════
%matplotlib inline

# results = sim.run_adaptive(
results = sim.run(
    solver_config=SOLVER_CONFIG,
    save_output=False,           # defer saving — we handle it below
    progress_callback=_progress_callback,
#     boundary_threshold=0.05,      # adapt if >5% of content at boundary
#     max_doublings=8,             # 0 = no adaptive doubling
#     points_per_segment=50,       # check every 50 output points
)

# ── Recovery cascade: try every place the partial results may have landed ───
if results is None:
    results = getattr(sim, '_accumulated_results', None)
    if results is not None:
        print('  Recovered partial results from sim._accumulated_results')
if results is None:
    results = getattr(sim, '_partial_results', None)
    if results is not None:
        print('  Recovered partial results from sim._partial_results')

if results is None:
    print('Simulation produced no results — nothing to save.')
else:
    # ── SAVE FIRST, print summary AFTER, so a print failure on degenerate
    # partial data cannot prevent the save. ─────────────────────────────────
    try:
        sim._diag_text = sim.reaction_rates.format_diagnostic(
            mean_n_i=(results['mean_n_i'][-1]
                      if len(results.get('mean_n_i', [])) > 0 else None))
    except Exception as exc:
        print(f'  diag_text build failed ({type(exc).__name__}: {exc})')
        sim._diag_text = ''
    try:
        sim._save_output(results, SOLVER_CONFIG)
    except Exception as exc:
        import traceback
        print(f'  _save_output raised {type(exc).__name__}: {exc}')
        traceback.print_exc()

    # Now the summary prints — each one guarded so a missing key cannot
    # mask failures elsewhere.
    def _safe(label, key, fmt='{:.4e}', mul=1.0):
        arr = results.get(key)
        if arr is None or len(arr) == 0:
            print(f'{label}: <unavailable>')
        else:
            try:
                print(f'{label}: ' + fmt.format(float(arr[-1]) * mul))
            except Exception as exc:
                print(f'{label}: <{type(exc).__name__}>')

    t_arr = results.get('t', [])
    print(f'\nFinal domain: I={sim.input_data.I}  V={sim.input_data.V}')
    if len(t_arr) > 0:
        print(f'Solution:         {len(t_arr)} time points, '
              f't = [{t_arr[0]:.2e}, {t_arr[-1]:.2e}] s')
    _safe('Final dose      ', 'dose')
    _safe('Swelling (final)', 'swelling', '{:.6f} %', 100.0)
    _safe('C_He_tot (final)', 'C_He_tot', '{:.3e}')
    _safe('mean_n_i (final)', 'mean_n_i', '{:.2f}')
    _safe('mean_n_v (final)', 'mean_n_v', '{:.2f}')
    _safe('N_loops  (final)', 'N_loops',  '{:.3e}')
    _safe('N_voids  (final)', 'N_voids',  '{:.3e}')
    _safe('delta_FP (final)', 'delta_FP', '{:.3e}  (Eq. 164)')
    _safe('delta_He (final)', 'delta_He', '{:.3e}  (Eq. 165)')

Applied 16 parameter overrides:
             T = 673
           eta = 0.3
        f_cl_i = 0.5
        f_cl_v = 0.15
        E_m_1D = 0.4
         L_hat = 5
           c_C = 0.001
     E_b_C_SIA = 0.65
         rho_d = 100000000000000.0
           Z_i = 1.08
          Z_ii = 1.01
    i_discrete = 100000
    v_discrete = 100000
  shape_function = linear
         I_bin = 0
         V_bin = 0

Full per-size system: N_eq = 200006

Launching solver_mode='active_window' …
C++ solver: solver.exe  N_eq=200006  solver_mode='active_window'  physics='full_CD_fission'


[OpenMP] auto-selected 24 threads for N_eq=200006 (hw_max=24)
[OpenMP_threads] 24
RadCluster_1_0 solver: N_eq=200006  physics_option=0  he_mode=0  he_kinetics=quasi_steady_state  C_floor=1e-25  window_mode=4  win_SIA=[0,9->99999]  win_VAC=[0,9->99999]
[cvode] pt=1/200  t=1.16232e-06  steps=7  rhs=10  nlcf=0  etf=0  h=1.63948e-07  ret=0
[cvode] pt=2/200  t=1.351e-06  steps=14  rhs=24  nlcf=0  etf=2  h=1.049e-07  ret=0
[cvode] pt=3/200  t=1.570e-06  steps=15  rhs=28  nlcf=0  etf=3  h=3.285e-07  ret=0
[cvode] pt=4/200  t=1.825e-06  steps=16  rhs=29  nlcf=0  etf=3  h=3.285e-07  ret=0
[cvode] pt=5/200  t=2.121e-06  steps=17  rhs=30  nlcf=0  etf=3  h=3.285e-07  ret=0
[cvode] pt=6/200  t=2.466e-06  steps=18  rhs=31  nlcf=0  etf=3  h=3.285e-07  ret=0
[cvode] pt=7/200  t=2.866e-06  steps=19  rhs=32  nlcf=0  etf=3  h=6.989e-07  ret=0
[cvode] pt=8/200  t=3.331e-06  steps=19  rhs=32  nlcf=0  etf=3  h=6.989e-07  ret=0
[cvode] pt=9/200  t=3.872e-06  steps=20  rhs=33  nlcf=0  etf=3  h=1.292e-06  ret=